<a href="https://colab.research.google.com/github/fcoliveira-utfpr/aquacrop_ml/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Início - Bibliotecas**
---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# **Preparando dataframe**
---

In [2]:
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/aquacrop_ml/refs/heads/main/produtividade_locais.csv"
df = pd.read_csv(url)

# Correção para o SyntaxError: remover o '.' final
cidade = df['Municipio']

df.drop(columns=['Municipio'], inplace=True)

# Identifica colunas que podem conter valores numéricos com vírgula como separador decimal
numeric_cols_potential = ['Latitude', 'Longitude', 'Altitude'] + [col for col in df.columns if str(col).isdigit()]

for col in numeric_cols_potential:
    if col in df.columns:
        # Substitui vírgulas por pontos e converte para numérico
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce') # 'coerce' transforma erros em NaN
df["Municipio"] = cidade
df_municipios = df
df_municipios.columns

Index(['Localidade', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       'Latitude', 'Longitude', 'Altitude', 'Municipio'],
      dtype='object')

In [3]:
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/aquacrop_ml/refs/heads/main/df_final.csv"
df = pd.read_csv(url)
df["Amp"] = df['Tmax'] - df['Tmin']
df["Pefetiva"] = df["Chuva"] / df['ETc']
df['T_DEF'] = df['Tmax'] * df['DEF']
df.columns

Index(['Fase', 'Tmin', 'Tmax', 'Tmed', 'Chuva', 'UR', 'ETc', 'ETR', 'ARM',
       'DEF', 'EXC', 'N', 'PP', 'Data semeadura', 'Data maturação', 'ISNA',
       'PA', 'Municipio', 'Ano', 'Amp', 'Pefetiva', 'T_DEF'],
      dtype='object')

In [4]:
df_yield = df_municipios.melt(
    id_vars=['Municipio','Latitude','Longitude','Altitude'],
    value_vars=[str(y) for y in range(2007, 2023)],
    var_name='Ano',
    value_name='Yield_obs'
)
df_yield['Ano'] = df_yield['Ano'].astype(int)
df_yield['lat_lon'] = df_yield['Latitude'] * df_yield['Longitude']
df_model = df.merge(df_yield, on=['Municipio','Ano'], how='left')
df_model['Data semeadura'] = pd.to_datetime(df_model['Data semeadura'])
df_model['Data maturação'] = pd.to_datetime(df_model['Data maturação'])
df_model['DOY_semeadura'] = df_model['Data semeadura'].dt.dayofyear
df_model['DOY_maturacao'] = df_model['Data maturação'].dt.dayofyear
df_model = df_model.drop(columns=['Data semeadura', 'Data maturação'])
df_model

,Fase,Tmin,Tmax,Tmed,Chuva,UR,ETc,ETR,ARM,DEF,...,Amp,Pefetiva,T_DEF,Latitude,Longitude,Altitude,Yield_obs,lat_lon,DOY_semeadura,DOY_maturacao
0,1,19.677647,28.567059,24.122353,88.12,79.221765,19.723534,16.972564,21.520444,2.750970,...,8.889412,4.467759,78.587117,-25.03,-53.38,712,2845.000000,1336.1014,36,135
1,2,19.263929,27.982857,23.623393,126.51,81.817857,75.026633,67.007595,31.991833,8.019037,...,8.718929,1.686201,224.395578,-25.03,-53.38,712,2845.000000,1336.1014,36,135
2,3,17.996667,28.714545,23.355606,90.06,78.406970,121.193702,85.338870,26.020932,35.854832,...,10.717879,0.743108,1029.555191,-25.03,-53.38,712,2845.000000,1336.1014,36,135
3,4,13.463636,23.200909,18.332273,174.17,84.022273,38.123266,33.827918,42.082942,4.295347,...,9.737273,4.568601,99.655963,-25.03,-53.38,712,2845.000000,1336.1014,36,135
4,1,19.685882,27.502353,23.594118,116.49,84.984118,18.810160,17.357626,21.520444,1.452534,...,7.816471,6.192930,39.948101,-25.03,-53.38,712,2845.000000,1336.1014,46,145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3835,4,10.490000,20.120909,15.305455,96.27,87.280000,27.188762,26.296740,42.804193,0.892022,...,9.630909,3.540801,17.948286,-24.70,-53.78,524,841.356709,1328.3660,74,173
3836,1,17.077647,27.398235,22.237941,106.86,82.940588,14.502544,12.166781,20.026222,2.335763,...,10.320588,7.368362,63.995785,-24.70,-53.78,524,841.356709,1328.3660,84,183
3837,2,15.715000,25.499286,20.607143,219.57,84.451429,53.217603,45.099749,32.193890,8.117854,...,9.784286,4.125890,206.999473,-24.70,-53.78,524,841.356709,1328.3660,84,183
3838,3,11.166667,20.646667,15.906667,151.54,85.715455,61.679847,55.110138,43.960000,6.569708,...,9.480000,2.456880,135.642579,-24.70,-53.78,524,841.356709,1328.3660,84,183


In [5]:
vars_fase = ['Tmin','Tmax','Tmed','Chuva','UR','ETc','ETR','ARM',
             'DEF','EXC','ISNA','PA','Amp','Pefetiva','Latitude',
             'Longitude','Altitude', 'lat_lon', 'T_DEF' ]

df_wide = df_model.pivot_table(
    index=['Municipio','Ano','Yield_obs','DOY_semeadura','DOY_maturacao'],
    columns='Fase',
    values=vars_fase
)
df_wide.columns = [f'{v}_F{f}' for v,f in df_wide.columns]
df_wide = df_wide.reset_index()
df_wide.drop(columns=['Municipio','Altitude_F2','Altitude_F3',
                      'Altitude_F4', 'Latitude_F2', 'Latitude_F3', 'Latitude_F4',
                      'Longitude_F2', 'Longitude_F3', 'Longitude_F4',
                      'lat_lon_F2', 'lat_lon_F3', 'lat_lon_F4'
                      ], inplace=True)
df_wide

,Ano,Yield_obs,DOY_semeadura,DOY_maturacao,ARM_F1,ARM_F2,ARM_F3,ARM_F4,Altitude_F1,Amp_F1,...,Tmed_F4,Tmin_F1,Tmin_F2,Tmin_F3,Tmin_F4,UR_F1,UR_F2,UR_F3,UR_F4,lat_lon_F1
0,2007,2845.000000,36,135,21.520444,31.991833,26.020932,42.082942,712.0,8.889412,...,18.332273,19.677647,19.263929,17.996667,13.463636,79.221765,81.817857,78.406970,84.022273,1336.1014
1,2007,2845.000000,46,145,21.520444,16.285581,33.495412,45.442250,712.0,7.816471,...,16.746364,19.685882,18.864643,16.837879,12.126364,84.984118,78.091071,82.152424,84.736818,1336.1014
2,2007,2845.000000,56,155,21.520444,33.897219,40.168584,45.139172,712.0,9.161765,...,13.865000,19.877647,18.169286,15.059394,9.022273,80.415882,78.600714,83.042727,83.632727,1336.1014
3,2007,2845.000000,64,163,15.701927,34.580531,47.240000,36.124281,712.0,9.136471,...,13.455682,18.767059,18.147500,14.321818,8.099091,80.303529,78.669643,84.495758,81.862273,1336.1014
4,2007,2845.000000,74,173,11.863121,40.701915,36.415454,25.062313,712.0,10.528235,...,16.299545,18.095882,17.113929,11.284545,11.075909,78.188824,81.520357,83.506667,83.072273,1336.1014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,2022,841.356709,46,145,20.026222,40.547367,43.960000,33.034518,524.0,12.262941,...,15.315000,21.745294,18.871429,17.204242,10.035455,65.154118,78.785714,84.335152,82.622727,1328.3660
956,2022,841.356709,56,155,20.026222,37.645351,39.112399,42.593935,524.0,9.396471,...,15.095455,22.072353,17.966071,15.113333,10.262727,76.972353,81.000000,84.649697,84.357727,1328.3660
957,2022,841.356709,64,163,16.203520,26.510089,33.765798,42.249184,524.0,9.845882,...,16.138409,20.364706,16.770000,13.357879,11.543636,78.160000,82.319643,84.206970,86.343636,1328.3660
958,2022,841.356709,74,173,18.552419,26.379903,43.960000,42.804193,524.0,11.267059,...,15.305455,17.438235,17.158214,11.808485,10.490000,78.885294,83.798929,84.279394,87.280000,1328.3660


In [6]:
df_wide.to_csv('df_wide.csv', index=False)

# **Rodando ML**
---

In [ ]:
corr = df_wide.corr(numeric_only=True)['Yield_obs'].drop('Yield_obs')

top10 = corr.abs().sort_values(ascending=False).head(20)

top10

,Yield_obs
UR_F1,0.256040
ISNA_F3,0.240807
Amp_F3,0.232867
Tmax_F1,0.225948
ETR_F3,0.217399
ARM_F3,0.214009
Tmed_F1,0.209612
Amp_F4,0.203957
Chuva_F3,0.197827
T_DEF_F1,0.191169


In [ ]:
X = df_wide[top10.index]
y = df_wide['Yield_obs']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print('R2:', r2)
print('RMSE:', rmse)
print('MAE:', mae)

R2: 0.5868847437436363
RMSE: 569.030768635878
MAE: 434.0495522145609
